## **Deep Q-Learning on Atari Pong Using Stable Baselines3**

This assignment requires training a Deep Q-Network (DQN) agent to play an Atari environment using Stable Baselines3 and Gymnasium, comparing CnnPolicy against MlpPolicy, tuning key hyperparameters (learning rate, gamma, batch size, and epsilon) across 10 experiments, and documenting the observed behavior of each configuration in a results table. A separate `play.py` script then loads the best trained model and evaluates it using a greedy policy.

This notebook covers the training side of that pipeline. It sets up the Atari Pong environment with frame stacking, defines a reusable training function, and runs 10 hyperparameter experiments, including a controlled comparison between a CNN policy and an MLP policy under identical hyperparameters (Experiments 2 and 3). Each experiment trains a model, logs episode rewards and lengths, and saves a summary used to build the final results table and evaluate which configuration performed best.

**Student:** Emmanuella Briggs  
**Environment:** ALE/Pong-v5 (Gymnasium, Arcade Learning Environment)  
**Library:** Stable Baselines3 (DQN)  
**Frame Stack:** 4 frames  
**Policies Compared:** CnnPolicy, MlpPolicy  
**Timesteps per Run:** 200,000  
**Experiments:** 10 hyperparameter configurations  
**Date:** July 2026

## Environment Setup

Installs the libraries needed to train on Atari: `stable-baselines3` (our DQN
implementation), `gymnasium[atari]` + `ale-py` (the Atari environment), and
`autorom` to download the licensed game ROMs (`AutoROM --accept-license`).

We use Stable Baselines3's DQN instead of writing our own, so we can focus on
tuning hyperparameters and comparing CNN vs MLP policies rather than debugging
an RL algorithm from scratch. ROMs must be downloaded before any environment
is created, or `gym.make()` fails.

In [2]:
!pip install -q "stable-baselines3[extra]" "gymnasium[atari]" ale-py autorom[accept-rom-license]
!AutoROM --accept-license

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.


## Verify ROM Installation

Quick sanity check that the Atari ROMs actually downloaded to the expected
folder, and counts how many ROM files are present.

We check this before building any environment so a missing-ROM error shows up
here, with a clear cause, rather than later as a confusing `gym.make()` failure.

In [3]:
import os
rom_dir = "/usr/local/lib/python3.12/dist-packages/AutoROM/roms"
print(os.path.exists(rom_dir))
print(len(os.listdir(rom_dir)) if os.path.exists(rom_dir) else "folder not found")

True
110


## Connect to Google Drive

Mounts Google Drive and creates a project folder with `logs/` and `videos/`
subfolders to store trained models, training logs, and gameplay recordings.

Colab's local disk is wiped when the session disconnects, so saving to Drive
instead means our trained models and results survive across sessions and
reconnects.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/dqn_pong_project"  # change if you'd like a different path
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/logs", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/videos", exist_ok=True)
print("Project dir:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/dqn_pong_project


## Imports & Custom Logging Callback

Imports the RL and environment libraries, registers Atari environments with
Gym, and sets our environment to `ALE/Pong-v5`.

Defines `RewardLoggerCallback`, a custom Stable Baselines3 callback that
records reward and length for every completed episode during training. We
built this ourselves because SB3's built-in logging only prints periodic
averages to the console — it doesn't give us the raw per-episode data we need
to compute our own summary stats and compare experiments afterward.

In [5]:
import json, time
import numpy as np
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, VecMonitor
from stable_baselines3.common.callbacks import BaseCallback

gym.register_envs(ale_py)

ENV_ID = "ALE/Pong-v5"

class RewardLoggerCallback(BaseCallback):
  """Collects per-episode reward/length during training."""
  def __init__(self, verbose=0):
    super().__init__(verbose)
    self.episode_rewards = []
    self.episode_lengths = []

  def _on_step(self) -> bool:
    for info in self.locals.get("infos", []):
      if "episode" in info:
        self.episode_rewards.append(float(info["episode"]["r"]))
        self.episode_lengths.append(int(info["episode"]["l"]))
    return True

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Reusable Training Function

Defines `train_agent()`, a single function that builds the environment, trains
a DQN agent with the given hyperparameters, saves the model, and logs a JSON
summary (reward, episode length, training time).

We built one reusable function instead of copy-pasting training code for each
experiment, so every run follows the exact same setup and only the
hyperparameters we're testing (`lr`, `gamma`, `batch_size`, `policy`, epsilon
values) change between calls — this is what makes our 10 experiments directly
comparable to each other.

`VecFrameStack(n_stack=4)` stacks the last 4 frames together so the agent can
perceive motion (e.g. ball direction), which a single frame can't show.

In [6]:
def train_agent(
    run_name,
    policy="CnnPolicy",
    lr=1e-4,
    gamma=0.99,
    batch_size=32,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.1,
    buffer_size=100_000,
    timesteps=200_000,
    n_envs=4,
    seed=0,
):
    """Trains a DQN agent with the given hyperparameters, saves the model +
    a JSON summary, and returns the summary dict."""

    vec_env = make_atari_env(ENV_ID, n_envs=n_envs, seed=seed)
    vec_env = VecFrameStack(vec_env, n_stack=4)
    vec_env = VecMonitor(vec_env)

    model = DQN(
        policy=policy,
        env=vec_env,
        learning_rate=lr,
        gamma=gamma,
        batch_size=batch_size,
        buffer_size=buffer_size,
        exploration_initial_eps=epsilon_start,
        exploration_final_eps=epsilon_end,
        exploration_fraction=epsilon_decay,
        verbose=1,
        seed=seed,
        device="cuda",
        tensorboard_log=f"{PROJECT_DIR}/logs"
    )

    callback = RewardLoggerCallback()
    start = time.time()
    model.learn(total_timesteps=timesteps, callback=callback, tb_log_name=run_name)
    duration = time.time() - start

    model_path = f"{PROJECT_DIR}/{run_name}_model.zip"
    model.save(model_path)

    rewards = callback.episode_rewards
    lengths = callback.episode_lengths
    summary = {
        "run_name": run_name,
        "env_id": ENV_ID,
        "policy": policy,
        "hyperparameters": {
            "lr": lr, "gamma": gamma, "batch_size": batch_size,
            "epsilon_start": epsilon_start, "epsilon_end": epsilon_end,
            "epsilon_decay": epsilon_decay, "buffer_size": buffer_size,
        },
        "timesteps": timesteps,
        "training_time_seconds": round(duration, 1),
        "num_episodes_completed": len(rewards),
        "mean_reward_last_10_episodes": round(float(np.mean(rewards[-10:])), 2) if rewards else None,
        "mean_reward_overall": round(float(np.mean(rewards)), 2) if rewards else None,
        "mean_episode_length_last_10": round(float(np.mean(lengths[-10:])), 2) if lengths else None,
    }

    summary_path = f"{PROJECT_DIR}/logs/{run_name}_summary.json"
    with open(summary_path, "w") as f:
      json.dump(summary, f, indent=2)

    print(f"\nSaved model to {model_path}")
    print(f"Saved summary to {summary_path}")
    print(json.dumps(summary, indent=2))
    return summary


## Experiment 1: Baseline CNN Run

Trains the first CNN policy configuration: `lr=2.5e-4, gamma=0.99, batch_size=32`,
with default epsilon settings.

I used this as a baseline run with a moderate learning rate to establish a
reference point before exploring higher/lower learning rates and larger batch
sizes in later experiments.

**Result:** mean reward (overall) = -20.82, mean reward (last 10 episodes) = -20.9.
The agent is still losing almost every point at this stage — expected this
early, since epsilon-greedy exploration is still active for part of training.

In [6]:
summary = train_agent(
    run_name="Emma_exp01",
    policy="CnnPolicy",
    lr=2.5e-4,
    gamma=0.99,
    batch_size=32,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.1,
    timesteps=200_000,
)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /content/drive/MyDrive/dqn_pong_project/logs/Emma_exp01_1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 210      |
|    ep_rew_mean      | -20.5    |
|    exploration_rate | 0.956    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 147      |
|    time_elapsed     | 6        |
|    total_timesteps  | 924      |
| train/              |          |
|    learning_rate    | 0.00025  |
|    loss             | 0.08     |
|    n_updates        | 51       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 206      |
|    ep_rew_mean      | -20.6    |
|    exploration_rate | 0.921    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 205      |
|    time_elapsed     | 8        |
|    total_timesteps  | 1664     |
| train/              |          |
|    learning_rate    | 0.00025  |
|    loss             | 0.0432   |
|    n_updates      

## Experiment 2: MLP Policy Run

Trains an MLP policy configuration: `lr=5e-4, gamma=0.99, batch_size=256`.

I chose these hyperparameters specifically so I could later run the same
exact configuration with a CNN policy (Experiment 3), giving me a controlled
comparison where policy architecture is the only variable that changes.

**Result:** mean reward (overall) = -20.94, mean reward (last 10 episodes) = -20.9,
mean episode length (last 10) = 192.0. The agent shows almost no improvement
over the course of training — flattening the pixel frames into a single vector
destroys the spatial information needed to track the ball and paddle.

In [7]:
summary = train_agent(
    run_name="Emma_exp02",
    policy="MlpPolicy",
    lr=5e-4,
    gamma=0.99,
    batch_size=256,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.1,
    timesteps=200_000,
)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cuda device
Wrapping the env in a VecTransposeImage.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Logging to /content/drive/MyDrive/dqn_pong_project/logs/Emma_exp02_2
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 206      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.957    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 190      |
|    time_elapsed     | 4        |
|    total_timesteps  | 908      |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 0.0331   |
|    n_updates        | 50       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 205      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.914    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 243      |
|    time_elapsed     | 7        |
|    total_timesteps  | 1808     |
| train/              |          |
|    learning_rate   

## Experiment 3: CNN Policy Run (MLP vs CNN Comparison)

Trains a CNN policy using the exact same hyperparameters as Experiment 2
(`lr=5e-4, gamma=0.99, batch_size=256`), changing only the policy architecture.

I set this up as a direct comparison against Experiment 2 to isolate the
effect of policy architecture alone, since every other setting is identical.

**Result:** mean reward (overall) = -18.93, mean reward (last 10 episodes) = -17.0,
mean episode length (last 10) = 471.2 — clearly outperforming the MLP run in
Experiment 2 on every metric. The CNN's convolutional layers pick up spatial
and motion patterns across the 4 stacked frames (ball position, paddle
position, direction of travel), which the MLP loses by flattening the image
into a single vector. This is also my best-performing experiment out of all 10.

In [7]:
summary = train_agent(
    run_name="Emma_exp03",
    policy="CnnPolicy",
    lr=5e-4,
    gamma=0.99,
    batch_size=256,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.1,
    timesteps=200_000,
)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /content/drive/MyDrive/dqn_pong_project/logs/Emma_exp03_5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 208      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.955    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 147      |
|    time_elapsed     | 6        |
|    total_timesteps  | 940      |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 0.0325   |
|    n_updates        | 52       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 206      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.913    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 193      |
|    time_elapsed     | 9        |
|    total_timesteps  | 1824     |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 0.0327   |
|    n_updates      

## Experiment 4: Higher Discount Factor (gamma=0.999)

Trains a CNN policy with `lr=1e-4, gamma=0.999, batch_size=32` increasing
gamma from the standard 0.99 to 0.999 while keeping everything else at
baseline values.

I tested this to see whether valuing future rewards more heavily (a higher
gamma) would help the agent learn longer-term strategy in rallies, compared
to the more standard 0.99 discount factor.

**Result:** mean reward (overall) = -20.86, essentially no improvement over
the baseline. A higher gamma alone doesn't help at this training budget;
Pong's short episodes and immediate point-scoring don't require valuing
rewards far into the future as much as longer-horizon games would.

In [ ]:
summary = train_agent(
    run_name="Emma_exp04",
    policy="CnnPolicy",
    lr=1e-4,
    gamma=0.999,
    batch_size=32,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.1,
    timesteps=200_000,
)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /content/drive/MyDrive/dqn_pong_project/logs/Emma_exp04_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 210      |
|    ep_rew_mean      | -20.5    |
|    exploration_rate | 0.956    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 220      |
|    time_elapsed     | 4        |
|    total_timesteps  | 924      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0802   |
|    n_updates        | 51       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 206      |
|    ep_rew_mean      | -20.6    |
|    exploration_rate | 0.921    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 253      |
|    time_elapsed     | 6        |
|    total_timesteps  | 1664    

## Experiment 5: Larger Batch Size (batch_size=128)

Trains a CNN policy with lr=1e-4, gamma=0.99, batch_size=128, increasing the
batch size from the baseline 32 while keeping other settings the same.

I tested this to see whether sampling more experiences per update would give
more stable gradient estimates and better learning than a smaller batch.

Result: mean reward (overall) = -20.63, a mild improvement over the baseline.
Larger batches appear to stabilize the Q-value updates somewhat, which
motivated testing an even larger batch size in the next experiment.

In [ ]:
summary = train_agent(
    run_name="Emma_exp05",
    policy="CnnPolicy",
    lr=1e-4,
    gamma=0.99,
    batch_size=128,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.1,
    timesteps=200_000,
)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /content/drive/MyDrive/dqn_pong_project/logs/Emma_exp05_1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 219      |
|    ep_rew_mean      | -20.5    |
|    exploration_rate | 0.954    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 175      |
|    time_elapsed     | 5        |
|    total_timesteps  | 972      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.06     |
|    n_updates        | 54       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 219      |
|    ep_rew_mean      | -20.4    |
|    exploration_rate | 0.909    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 233      |
|    time_elapsed     | 8        |
|    total_timesteps  | 1924     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0526   |
|    n_updates      

## Experiment 6: Even Larger Batch Size (batch_size=256)

Trains a CNN policy with lr=1e-4, gamma=0.99, batch_size=256, increasing the
batch size further from 128 to 256 while keeping other settings the same.

I tested this to follow up on the improvement seen in Experiment 5, checking
whether an even larger batch size would continue to help or start to plateau.

Result: mean reward (overall) = -19.86, better than both the baseline and
Experiment 5. Larger batches continued to help stabilize learning, making
this my second best performing experiment after Experiment 3.

In [ ]:
summary = train_agent(
    run_name="Emma_exp06",
    policy="CnnPolicy",
    lr=1e-4,
    gamma=0.99,
    batch_size=256,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.1,
    timesteps=200_000,
)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /content/drive/MyDrive/dqn_pong_project/logs/Emma_exp06_2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 208      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.955    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 178      |
|    time_elapsed     | 5        |
|    total_timesteps  | 940      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0325   |
|    n_updates        | 52       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 207      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.911    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 199      |
|    time_elapsed     | 9        |
|    total_timesteps  | 1868     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0391   |
|    n_updates      

## Experiment 7: Lower Learning Rate with Large Batch (lr=5e-5, batch_size=256)

Trains a CNN policy with lr=5e-5, gamma=0.99, batch_size=256, keeping the
batch size from Experiment 6 but halving the learning rate.

I tested this to see whether a smaller learning rate paired with the large
batch size from Experiment 6 would learn more slowly but end up more stable
or precise.

Result: mean reward (overall) = -20.41, slightly worse than Experiment 6.
The lower learning rate slowed down learning without a large enough
timestep budget to catch up, so batch_size=256 combined with the original
lr=1e-4 remains the better combination.

In [ ]:
summary = train_agent(
    run_name="Emma_exp07",
    policy="CnnPolicy",
    lr=5e-5,
    gamma=0.99,
    batch_size=256,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.1,
    timesteps=200_000,
)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /content/drive/MyDrive/dqn_pong_project/logs/Emma_exp07_4


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 208      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.955    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 156      |
|    time_elapsed     | 5        |
|    total_timesteps  | 940      |
| train/              |          |
|    learning_rate    | 5e-05    |
|    loss             | 0.0325   |
|    n_updates        | 52       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 207      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.911    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 205      |
|    time_elapsed     | 9        |
|    total_timesteps  | 1868     |
| train/              |          |
|    learning_rate    | 5e-05    |
|    loss             | 0.0391   |
|    n_updates      

## Experiment 8: Faster Epsilon Decay (epsilon_decay=0.05)

Trains a CNN policy with lr=1e-4, gamma=0.99, batch_size=32, reducing
epsilon_decay from the baseline 0.1 to 0.05 so exploration drops off faster.

I tested this to see whether shifting from exploration to exploitation
earlier in training would help the agent exploit a good policy sooner.

Result: mean reward (overall) = -20.79, no meaningful change from the
baseline. Decaying exploration faster didn't help at this training budget,
suggesting the agent needs the full exploration period to gather enough
experience before exploitation becomes useful.

In [ ]:
summary = train_agent(
    run_name="Emma_exp08",
    policy="CnnPolicy",
    lr=1e-4,
    gamma=0.99,
    batch_size=32,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.05,
    timesteps=200_000,
)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /content/drive/MyDrive/dqn_pong_project/logs/Emma_exp08_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 215      |
|    ep_rew_mean      | -20.5    |
|    exploration_rate | 0.9      |
| time/               |          |
|    episodes         | 4        |
|    fps              | 319      |
|    time_elapsed     | 3        |
|    total_timesteps  | 1052     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0828   |
|    n_updates        | 59       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 211      |
|    ep_rew_mean      | -20.6    |
|    exploration_rate | 0.812    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 350      |
|    time_elapsed     | 5        |
|    total_timesteps  | 1984    

## Experiment 9: Slower Epsilon Decay (epsilon_decay=0.3)

Trains a CNN policy with lr=1e-4, gamma=0.99, batch_size=32, increasing
epsilon_decay from the baseline 0.1 to 0.3 so exploration lasts longer.

I tested this as the opposite of Experiment 8, checking whether extending
exploration would help the agent discover a better policy before switching
to exploitation.

Result: mean reward (overall) = -20.76, no meaningful change from the
baseline. Extending exploration didn't help either, showing that
epsilon_decay has little impact on performance at this training budget
compared to changes in batch_size.

In [ ]:
summary = train_agent(
    run_name="Emma_exp09",
    policy="CnnPolicy",
    lr=1e-4,
    gamma=0.99,
    batch_size=32,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.3,
    timesteps=200_000,
)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /content/drive/MyDrive/dqn_pong_project/logs/Emma_exp09_3


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 205      |
|    ep_rew_mean      | -20.8    |
|    exploration_rate | 0.992    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 155      |
|    time_elapsed     | 5        |
|    total_timesteps  | 888      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0403   |
|    n_updates        | 49       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 208      |
|    ep_rew_mean      | -20.9    |
|    exploration_rate | 0.984    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 203      |
|    time_elapsed     | 8        |
|    total_timesteps  | 1688     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0699   |
|    n_updates      

## Experiment 10: Combining Higher Gamma and Larger Batch Size

Trains a CNN policy with lr=1e-4, gamma=0.999, batch_size=256, combining the
higher gamma from Experiment 4 with the larger batch size from Experiment 6.

I tested this to see whether combining two settings that each looked
reasonable on their own would produce a better result than either alone.

Result: mean reward (overall) = -20.35, worse than Experiment 6's -19.86.
Combining gamma=0.999 with batch_size=256 underperformed batch_size=256 on
its own, showing that batch_size was the more useful change, and increasing
gamma on top of it did not help further.

In [ ]:
summary = train_agent(
    run_name="Emma_exp10",
    policy="CnnPolicy",
    lr=1e-4,
    gamma=0.999,
    batch_size=256,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.1,
    timesteps=200_000,
)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /content/drive/MyDrive/dqn_pong_project/logs/Emma_exp10_1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 208      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.955    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 170      |
|    time_elapsed     | 5        |
|    total_timesteps  | 940      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0326   |
|    n_updates        | 52       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 207      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.911    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 182      |
|    time_elapsed     | 10       |
|    total_timesteps  | 1868     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0391   |
|    n_updates      

## Hyperparameter Experiments Summary

The assignment requires testing 10 different hyperparameter combinations and
recording the observed behavior of each. The table below summarizes all 10
experiments run in this notebook, including the CNN vs MLP comparison pair
(Experiments 2 and 3), using the actual results logged during training.

| Experiment | Policy | Learning Rate | Gamma | Batch Size | Epsilon Decay | Mean Reward (Overall) | Noted Behavior |
|---|---|---|---|---|---|---|---|
| Emma_exp01 | CNN | 2.5e-4 | 0.99 | 32 | 0.1 | -20.82 | Baseline run, still losing almost every point at this stage |
| Emma_exp02 | MLP | 5e-4 | 0.99 | 256 | 0.1 | -20.94 | Almost no improvement, flattening pixels loses spatial information |
| Emma_exp03 | CNN | 5e-4 | 0.99 | 256 | 0.1 | -18.93 | Best performing run, clearly outperforms the matching MLP run in exp02 |
| Emma_exp04 | CNN | 1e-4 | 0.999 | 32 | 0.1 | -20.86 | Higher gamma alone gives no improvement over baseline |
| Emma_exp05 | CNN | 1e-4 | 0.99 | 128 | 0.1 | -20.63 | Larger batch size gives a mild improvement |
| Emma_exp06 | CNN | 1e-4 | 0.99 | 256 | 0.1 | -19.86 | Larger batch size continues to help, second best overall |
| Emma_exp07 | CNN | 5e-5 | 0.99 | 256 | 0.1 | -20.41 | Lower learning rate slows learning, slightly worse than exp06 |
| Emma_exp08 | CNN | 1e-4 | 0.99 | 32 | 0.05 | -20.79 | Faster epsilon decay gives no meaningful change |
| Emma_exp09 | CNN | 1e-4 | 0.99 | 32 | 0.3 | -20.76 | Slower epsilon decay gives no meaningful change |
| Emma_exp10 | CNN | 1e-4 | 0.999 | 256 | 0.1 | -20.35 | Combining higher gamma and larger batch underperforms batch size alone |

## Summary of Findings

The CNN vs MLP comparison (Experiments 2 and 3) showed a clear result: CNN
outperformed MLP under identical hyperparameters, since convolutional layers
preserve the spatial and motion information that MLP loses by flattening
pixels.

Among hyperparameters, batch size had the biggest impact, with performance
improving as batch size increased from 32 to 256 (Experiments 1, 5, 6).
Gamma, learning rate, and epsilon decay showed little to no effect on their
own. The best performing configuration overall was Experiment 3
(CnnPolicy, lr=5e-4, gamma=0.99, batch_size=256), with a mean reward of -18.93.